# Lab: Implement the Attention Equation
Implement the attention equation myself.
Use the learned projection matrices  WQ ,  WK ,  WV  from the Gemma-1B model and compute the attention weights and the new contextual embedding.

## Purpose:
- Use projection matrices to compute the attention weights and the output of the attention mechanism.
- Removing/replacing markup tags
- Using Unicode categories

### Topics:
- Attention weights
- Projection matrices
- Contextual embeddings

### Steps
- Load the pre-trained Gemma-1B model and use this model to compute query, key, and value matrices.
- Implement the attention equation.
- Visualize and compare your attention weights with those of the reference implementation.

Date: 2026-02-26

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_4/gdm_lab_4_2_implement_attention_equation.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install orbax-checkpoint==0.11.21
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import jax # For working with vectors and matrices.
import jax.numpy as jnp # For working with vectors and matrices.
from ai_foundations import generation # For prompting the Gemma model.
from ai_foundations import visualizations # For visualizing attention weights.
from ai_foundations import attention # For working with Q,K,V matrices.
# For providing feedback on your implementation.
from ai_foundations.feedback.course_4 import attention as attention_feedback

## Load the model and visualize attention weights

Load the Gemma-1B parameters.

Initialize a cache to speed up the visualizations when adjusting the layer parameter.

In [ ]:
# Initialize caches for prompts.
previous_prompt = None
previous_prompt2 = None

# Load special version of the Gemma-1B model that provides access to attention
# weights and QKV matrices.
print("Loading Gemma-1B...")
model = generation.load_gemma("Gemma-1B-AttentionWeight")
print("Loaded Gemma-1B.")

In [ ]:
# @title Visualize attention weights (reference implementation)
# NOTE: This block fails because the typeguard library removed the 'memo' argument in version 3.0.
# The gemma library in this notebook is still trying to use this argument with a newer version of typeguard, leading to this error.
layer = 19  # @param {type:"slider", min: 0, max: 25}

prompt = "Jide was hungry so she went looking for"  # @param {type: "string"}
# @markdown Check the following box to display the attention weights for all tokens, not just for the generated one:
show_all_weights = True  # @param {type:"boolean"}


if prompt != previous_prompt:
  (
      output_text,
      _,
      tokenizer,
      attention_weights,
      _,
      qkv_dict,
  ) = generation.prompt_attention_transformer_model(
      prompt, model, sampling_mode="greedy"
  )
  tokens = [tokenizer.tokens[t] for t in tokenizer.encode(output_text)]
  previous_prompt = prompt

print(f"Generated text: {output_text}")

visualizations.visualize_attention(
    tokens,
    attention_weights[f"layer_{layer}"],
    layer,
    min_line_thickness=0,
    max_line_thickness=5,
    show_all_weights=show_all_weights,
)

## Attention equation

------
>Implement the attention mechanism for a specific layer. The cell already makes a call to `get_qkv_matrices` which contains the query, key, and value projections for every token in the sentence.
>
>1. Inspect the query, key, and value projection matrices. What is their dimension?
>
>2. Compute $d_k$, the dimension of the key vectors which you need to compute the normalization factor. Use [`.shape`](https://docs.jax.dev/en/latest/_autosummary/jax.Array.shape.html) for this.
>
> 3. Compute the logits by performing a matrix multiplication between the query projections and the key projections and normalizing them by the square root of $d_k$:
>
>    $$\mbox{logits} = \frac{QK^T}{\sqrt{d_k}}$$
>
>    Instead of `jnp.matmul`, you can also use the [`@` operator](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.matmul.html) to multiply two matrices. For example, `A @ B` computes the matrix product between matrices `A` and `B`. To compute the square root, use the function `jnp.sqrt`. Finally, transpose a matrix `A` by calling [`A.T`](https://docs.jax.dev/en/latest/_autosummary/jax.Array.T.html).
>
> 4. Compute the attention weights `alpha` by applying the SoftMax function to the logits. You can either implement your own SoftMax function or use [`jax.nn.softmax`](https://docs.jax.dev/en/latest/_autosummary/jax.nn.softmax.html):
>
> $$\alpha = \mbox{SoftMax}(\mbox{logits})$$
>
>5. Compute the output of the attention mechanism by multiplying the attention weights $\alpha$ with the value projections:
>$$Y = \alpha V$$
>
>Once you have implemented the computations by filling in the missing parts of the next cell, run the test cell and the "Visualize attention weights (your implementation)" cell to see the attention weights.
------

In [ ]:
def compute_attention(
    qkv_dict: dict[str, dict[str, jax.Array]], layer: int
) -> tuple[jax.Array, jax.Array, jax.Array]:
  """Computes the attention weights for a layer.

  Args:
    qkv_dict: A dictionary containing the raw query, key, and value
      projections for all layers. The keys are strings identifying the layers
      and matrix types, and values are the corresponding JAX arrays.
    layer: The specific layer for which to compute the attention weights.

  Returns:
    Y: The output of the attention layer. Shape: (n_tokens, embedding_dim).
    alpha: The attention weights. Shape: (n_tokens, n_tokens).
    logits: The raw logits used to compute the attention weights.
      Shape: (n_tokens, n_tokens).
  """
  # Extract the query, key, and value projection matrices from `qkv_dict`.
  query_proj, key_proj, value_proj = attention.get_qkv_matrices(qkv_dict, layer)

  logits = query_proj * key_proj.T / jnp.sqrt(dim_key)

  alpha = softmax(logits)

  Y = alpha * value_proj

  return Y, alpha, logits